# RoamAI — PySpark analytics pipelineReads the `activities` table from Lakebase via Spark JDBC, computesaggregate metrics (activity counts, average duration, weather sensitivitybreakdown), and writes the results back as an `activity_analytics` table.This is the "data pipeline in Spark" for the project. Uses standardPySpark DataFrame operations that would scale to millions of rows ifthe dataset grew.## Prerequisites- Lakebase URL stored in the `database/lakebase-url` secret (already set up)- Activities table populated with the 15 Kauai seed rows

## 1. Load Lakebase connection details from the secret

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

# Fetch and decode the Lakebase URL (stored as base64)
w = WorkspaceClient()
secret = w.secrets.get_secret(scope="database", key="lakebase-url")
LAKEBASE_URL = base64.b64decode(secret.value).decode("utf-8")

# Parse into components for Spark JDBC
parsed = urlparse(LAKEBASE_URL)
JDBC_URL = f"jdbc:postgresql://{parsed.hostname}:{parsed.port}{parsed.path}"
JDBC_USER = parsed.username
JDBC_PASS = parsed.password

print(f"JDBC URL: jdbc:postgresql://{parsed.hostname}:{parsed.port}{parsed.path}")
print(f"User:     {JDBC_USER}")
print(f"Password: {'*' * 12}")

## 2. Read the activities table via Spark JDBC

In [0]:
activities_df = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "activities")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
)

# Drop the vector column so we can display cleanly (Spark can't render vectors)
activities_df = activities_df.drop("description_embedding")

print(f"Loaded {activities_df.count()} activity rows")
activities_df.printSchema()

## 3. Preview the raw data

In [0]:
activities_df.select(
    "id", "name", "category", "weather_sensitive", "duration_hours"
).show(20, truncate=False)

## 4. Aggregation: activities by categoryClassic groupBy + aggregation. Counts and average duration per category,plus the count of weather-sensitive vs weather-safe activities.

In [0]:
from pyspark.sql import functions as F

by_category = (
    activities_df
    .groupBy("category")
    .agg(
        F.count("*").alias("activity_count"),
        F.round(F.avg("duration_hours"), 2).alias("avg_duration_hrs"),
        F.round(F.sum("duration_hours"), 2).alias("total_duration_hrs"),
        F.sum(F.col("weather_sensitive").cast("int")).alias("outdoor_count"),
        F.sum((~F.col("weather_sensitive")).cast("int")).alias("indoor_count"),
    )
    .orderBy(F.desc("activity_count"))
)

by_category.show(truncate=False)

## 5. Aggregation: weather sensitivity per destinationGroups by destination_id and computes what percentage of activities areweather-sensitive. Useful signal for the agent's rescheduling logic:destinations with more outdoor activities benefit more fromweather-aware planning.

In [0]:
by_destination = (
    activities_df
    .groupBy("destination_id")
    .agg(
        F.count("*").alias("total_activities"),
        F.sum(F.col("weather_sensitive").cast("int")).alias("outdoor_activities"),
        F.round(
            100.0 * F.sum(F.col("weather_sensitive").cast("int")) / F.count("*"),
            1,
        ).alias("pct_weather_sensitive"),
        F.round(F.avg("duration_hours"), 2).alias("avg_duration_hrs"),
        F.collect_set("category").alias("categories"),
    )
    .orderBy(F.desc("total_activities"))
)

by_destination.show(truncate=False)

## 6. Combine into a single analytics DataFrame to persistWraps both aggregations into one wide-form analytics table with a `dimension`column marking whether each row is a per-category or per-destination summary.

In [0]:
from pyspark.sql.types import StringType

analytics_by_category = (
    by_category
    .withColumn("dimension", F.lit("category"))
    .withColumn("dimension_value", F.col("category"))
    .withColumn("outdoor_pct",
                F.round(100.0 * F.col("outdoor_count") / F.col("activity_count"), 1))
    .select(
        "dimension",
        "dimension_value",
        F.col("activity_count").alias("total_activities"),
        F.col("outdoor_count").alias("outdoor_activities"),
        F.col("outdoor_pct").alias("pct_weather_sensitive"),
        "avg_duration_hrs",
    )
)

analytics_by_destination = (
    by_destination
    .withColumn("dimension", F.lit("destination"))
    .withColumn("dimension_value", F.col("destination_id").cast(StringType()))
    .select(
        "dimension",
        "dimension_value",
        "total_activities",
        "outdoor_activities",
        "pct_weather_sensitive",
        "avg_duration_hrs",
    )
)

analytics = analytics_by_category.unionByName(analytics_by_destination)

print(f"Combined analytics rows: {analytics.count()}")
analytics.show(truncate=False)

## 7. Write results back to LakebasePersists the analytics DataFrame as a new `activity_analytics` tablevia Spark JDBC. Uses `overwrite` mode so re-running the pipelinereplaces prior results.

In [0]:
(
    analytics.write
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "activity_analytics")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)

print("Wrote analytics table back to Lakebase")

## 8. Verify — read the analytics table backReads the freshly-written table via Spark JDBC to confirm the round tripworked. Should show the same rows we wrote.

In [0]:
verify_df = (
    spark.read
    .format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", "activity_analytics")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print(f"Read back {verify_df.count()} analytics rows")
verify_df.orderBy("dimension", F.desc("total_activities")).show(truncate=False)

## Pipeline summaryThis notebook demonstrates a complete Spark data pipeline over the RoamAIactivities dataset:1. **Extract** — read from Lakebase Postgres via Spark JDBC2. **Transform** — two aggregations using PySpark DataFrame ops (groupBy,   agg, avg, sum, round, cast, collect_set, union)3. **Load** — write back to Lakebase as a new `activity_analytics` tableIn production, this notebook would run on a scheduled Databricks Workflow(daily or weekly) to keep the analytics table fresh as new activitiesare added. The Streamlit dashboard or agent could then read`activity_analytics` for quick summary stats without re-computing overthe full activities table.